In [ ]:
import os
import cv2
import random
import numpy as np


healthy_dir = "/Healthy" #Provide your directory for healthy images
substance_dir= "/data2/suracepkr/FAS"  #Provide your directory for Foreign attached substances images
output_dir = "/data2/suracepkr/augmented_images"  #Provide your directory for augmented images

image_dir = os.path.join(output_dir, "images")
label_dir = os.path.join(output_dir, "labels")

copies_per_image = 1
class_id = 0
seed = 42

apply_resize = False
apply_rotation = False

min_scale = 0.25
max_scale = 0.40
max_rot = 15

white_thresh = 245
placement_tries = 200
green_ratio_threshold = 0.60
margin = 10
blend_band_width = 12

In [ ]:
def set_seed(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)


def make_dirs():
    os.makedirs(image_dir, exist_ok=True)
    os.makedirs(label_dir, exist_ok=True)


def list_images(folder):
    exts = (".jpg", ".jpeg", ".png")
    return [f for f in os.listdir(folder) if f.lower().endswith(exts)]

In [ ]:
#processing of foreign attached substances images
def load_fas_remove_white(path, thresh=245, pad=2):
    img = cv2.imread(path)
    if img is None:
        return None, None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = (gray < thresh).astype(np.uint8) * 255

    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return None, None

    x1, y1 = xs.min(), ys.min()
    x2, y2 = xs.max(), ys.max()

    x1 = max(0, x1 - pad)
    y1 = max(0, y1 - pad)
    x2 = min(img.shape[1], x2 + pad + 1)
    y2 = min(img.shape[0], y2 + pad + 1)

    return img[y1:y2, x1:x2], mask[y1:y2, x1:x2]


def maybe_resize(patch, mask, bg_w):
    if not apply_resize:
        return patch, mask

    h, w = patch.shape[:2]
    target_w = int(bg_w * random.uniform(min_scale, max_scale))
    scale = target_w / max(w, 1)

    new_w = max(10, int(w * scale))
    new_h = max(10, int(h * scale))

    patch = cv2.resize(patch, (new_w, new_h))
    mask = cv2.resize(mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)

    return patch, mask


def maybe_rotate(patch, mask):
    if not apply_rotation:
        return patch, mask

    h, w = patch.shape[:2]
    angle = random.uniform(-max_rot, max_rot)

    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)

    patch_r = cv2.warpAffine(patch, M, (w, h), borderValue=(255, 255, 255))
    mask_r = cv2.warpAffine(mask, M, (w, h))

    return patch_r, mask_r

In [ ]:
def get_green_mask(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    lower = np.array([25, 20, 20])
    upper = np.array([95, 255, 255])

    mask = cv2.inRange(hsv, lower, upper)

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    return mask


def find_position(bg, green_mask, obj_mask, h, w):
    H, W = bg.shape[:2]

    obj_area = np.count_nonzero(obj_mask)
    if obj_area == 0:
        return None

    for _ in range(placement_tries):
        x = random.randint(margin, W - w - margin)
        y = random.randint(margin, H - h - margin)

        region = green_mask[y:y + h, x:x + w]
        overlap = np.logical_and(region > 0, obj_mask > 0)

        if overlap.sum() / obj_area >= green_ratio_threshold:
            return x, y

    return None

In [ ]:
#blending function
def blend(bg, patch, mask, x, y):
    h, w = patch.shape[:2]
    roi = bg[y:y + h, x:x + w]

    alpha = cv2.GaussianBlur(mask.astype(float) / 255, (5, 5), 0)
    alpha = alpha[..., None]

    blended = patch * alpha + roi * (1 - alpha)

    out = bg.copy()
    out[y:y + h, x:x + w] = blended.astype(np.uint8)

    return out

In [ ]:
def main():
    set_seed(seed)
    make_dirs()

    healthy_files = list_images(healthy_dir)
    fas_files = list_images(substance_dir)

    count = 0

    for hfile in healthy_files:
        bg = cv2.imread(os.path.join(healthy_dir, hfile))
        if bg is None:
            continue

        H, W = bg.shape[:2]
        green_mask = get_green_mask(bg)

        for _ in range(copies_per_image):
            sfile = random.choice(fas_files)

            patch, mask = load_fas_remove_white(
                os.path.join(substance_dir, sfile),
                white_thresh
            )
            if patch is None:
                continue

            patch, mask = maybe_resize(patch, mask, W)
            patch, mask = maybe_rotate(patch, mask)

            if patch is None:
                continue

            h, w = patch.shape[:2]
            pos = find_position(bg, green_mask, mask, h, w)
            if pos is None:
                continue

            x, y = pos
            result = blend(bg, patch, mask, x, y)

            img_name = f"aug_{count:05d}.jpg"
            cv2.imwrite(os.path.join(image_dir, img_name), result)

            print("saved:", img_name)
            count += 1

    print("done:", count)


if __name__ == "__main__":
    main()